# Final feature matrix

For a feature CSV, keep only the features that are

1. **reproducible**: ICC(2,1) >= `ICC_THRESHOLD` in every MRI channel (from `reproducibility_analysis.ipynb`), and
2. **not redundant**: |Spearman rho| < `REDUNDANCY_THRESHOLD` with every other kept feature
   (of a redundant group, the most reproducible feature is kept).

The selected features are saved as `OUTPUT_DIR/<csv name>`, and the decision for every feature as
`OUTPUT_DIR/selection_<csv name>`.

In [1]:
import sys
from pathlib import Path

CODE_ROOT = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline")
for folder in ["", "scripts", "pre_processing", "feature_extraction", "post_processing"]:
    if str(CODE_ROOT / folder) not in sys.path:
        sys.path.insert(0, str(CODE_ROOT / folder))

from pathlib import Path
from IPython.display import display
import pandas as pd
from final_features import select_final_features

FEATURE_DIR = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/outputs/features/pyradiomics")
CSV_PATH = FEATURE_DIR / "features_raw_image.csv"     # <- feature CSV to select from
ICC_DIR = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/outputs/icc")   # ICC tables written by ICC_check.ipynb
ICC_CSV = ICC_DIR / CSV_PATH.stem / "icc_per_channel.csv"   # ICC of the features in CSV_PATH
OUTPUT_DIR = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/outputs/features/final")
import yaml
config_path = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/config_files/setup.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config
ICC_THRESHOLD = config["THRESHOLD_REPRODUCIBILITY"]        # keep features with ICC >= this
REDUNDANCY_THRESHOLD = config["THRESHOLD_REDUNDANCY"]    # drop features with |rho| >= this to an already kept feature
KEEP_UNTESTED = False          # features without ICC (Gabor, Laws): False = drop, True = keep

Select the features of one CSV

In [2]:
report = select_final_features(CSV_PATH, ICC_CSV, OUTPUT_DIR, ICC_THRESHOLD, REDUNDANCY_THRESHOLD, KEEP_UNTESTED)

features_raw_image.csv: 107 features | 0 without ICC | 91 reproducible (ICC >= 0.75) | 78 selected after removing redundancy (|rho| >= 0.995)


Selected features

In [3]:
display(report[report["selected"]].sort_values("icc2_1", ascending=False))

,feature,icc2_1,constant,reproducible,redundant_with,selected
6,original_shape_Maximum2DDiameterSlice,0.999816,False,True,NaN,True
12,original_shape_SurfaceVolumeRatio,0.991447,False,True,NaN,True
2,original_shape_LeastAxisLength,0.985746,False,True,NaN,True
29,original_firstorder_TotalEnergy,0.984882,False,True,NaN,True
10,original_shape_Sphericity,0.984589,False,True,NaN,True
...,...,...,...,...,...,...
96,original_gldm_LargeDependenceHighGrayLevelEmph...,0.763371,False,True,NaN,True
17,original_firstorder_Entropy,0.762630,False,True,NaN,True
97,original_gldm_LargeDependenceLowGrayLevelEmphasis,0.755504,False,True,NaN,True
44,original_glcm_Idn,0.750913,False,True,NaN,True


Reproducible features removed because they are redundant

In [4]:
display(report[report["reproducible"] & ~report["selected"]])

,feature,icc2_1,constant,reproducible,redundant_with,selected
16,original_firstorder_Energy,0.984882,False,True,original_firstorder_TotalEnergy,False
27,original_firstorder_RootMeanSquared,0.928346,False,True,original_firstorder_Mean,False
41,original_glcm_Id,0.816932,False,True,original_glcm_Idm,False
48,original_glcm_JointAverage,0.798780,False,True,original_glcm_SumAverage,False
54,original_glcm_SumEntropy,0.750322,False,True,original_firstorder_Entropy,False
55,original_glcm_SumSquares,0.966909,False,True,original_glcm_ClusterTendency,False
66,original_glrlm_RunLengthNonUniformityNormalized,0.839833,False,True,original_glrlm_ShortRunEmphasis,False
71,original_glrlm_ShortRunLowGrayLevelEmphasis,0.957849,False,True,original_glrlm_LowGrayLevelRunEmphasis,False
76,original_glszm_LargeAreaEmphasis,0.785719,False,True,original_glszm_LargeAreaLowGrayLevelEmphasis,False
82,original_glszm_SmallAreaEmphasis,0.833397,False,True,original_glszm_SizeZoneNonUniformityNormalized,False


In [5]:
summary = []
for csv_path in sorted(FEATURE_DIR.glob("features_*.csv")):
    r = select_final_features(csv_path, ICC_DIR / csv_path.stem / "icc_per_channel.csv", OUTPUT_DIR, ICC_THRESHOLD, REDUNDANCY_THRESHOLD, KEEP_UNTESTED)
    summary.append({"file": csv_path.name, "features": len(r), "without ICC": int(r["icc2_1"].isna().sum()),
                    "reproducible": int(r["reproducible"].sum()), "selected": int(r["selected"].sum())})

summary = pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR / "selection_summary.csv", index=False)
summary

features_binary_patterns.csv: 279 features | 2 without ICC | 238 reproducible (ICC >= 0.75) | 166 selected after removing redundancy (|rho| >= 0.995)
features_exponential_filter.csv: 93 features | 0 without ICC | 78 reproducible (ICC >= 0.75) | 68 selected after removing redundancy (|rho| >= 0.995)
features_gabor_filter.csv: 372 features | 0 without ICC | 353 reproducible (ICC >= 0.75) | 290 selected after removing redundancy (|rho| >= 0.995)
features_gradient_filter.csv: 93 features | 0 without ICC | 75 reproducible (ICC >= 0.75) | 66 selected after removing redundancy (|rho| >= 0.995)
features_laws_filter.csv: 558 features | 0 without ICC | 524 reproducible (ICC >= 0.75) | 433 selected after removing redundancy (|rho| >= 0.995)
features_log_filter.csv: 465 features | 0 without ICC | 444 reproducible (ICC >= 0.75) | 370 selected after removing redundancy (|rho| >= 0.995)
features_logarithmic_filter.csv: 93 features | 0 without ICC | 71 reproducible (ICC >= 0.75) | 61 selected after re

,file,features,without ICC,reproducible,selected
0,features_binary_patterns.csv,279,2,238,166
1,features_exponential_filter.csv,93,0,78,68
2,features_gabor_filter.csv,372,0,353,290
3,features_gradient_filter.csv,93,0,75,66
4,features_laws_filter.csv,558,0,524,433
5,features_log_filter.csv,465,0,444,370
6,features_logarithmic_filter.csv,93,0,71,61
7,features_raw_image.csv,107,0,91,78
8,features_square_filter.csv,93,0,81,71
9,features_square_root_filter.csv,93,0,73,62
